In [94]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import InMemorySaver

In [95]:
load_dotenv()


True

In [96]:
google_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.9
)

openai_model = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0.9
)

In [97]:
from google import genai

In [98]:
client = genai.Client()

In [ ]:
interaction =client.interactions.create(
    model="gemini-3.5-flash-lite",
    input="Explain the demand for rust programming language in near future compared to other prominent language"
)

In [ ]:
class JokeState(TypedDict):

    topic: str
    joke_open: str
    joke_google: str
    exp_open: str
    exp_google: str


    

In [ ]:
def generate_joke(state: JokeState):

    prompt = f'Generate a joke on the topic {state["topic"]}'
    response_google = google_model.invoke(prompt).content
    response_open = openai_model.invoke(prompt).content

    return {'joke_open': response_open, "joke_google": response_google}

In [ ]:
def generate_explanation(state: JokeState):
    prompt_open = f'write an explanation for the joke - {state["joke_open"]}'
    prompt_google = f'write an explanation for the joke - {state["joke_google"]}'
    response_google = google_model.invoke(prompt_google).content
    response_open = openai_model.invoke(prompt_open).content

    return {'exp_open': response_open, 'exp_google': response_google}

In [ ]:
graph = StateGraph(JokeState)

In [ ]:
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

In [ ]:
workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

/home/j/miniconda3/envs/langgraph-test/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/j/miniconda3/envs/langgraph-test/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'topic': 'Football',
 'joke_open': 'Why did the football team bring string to the game?\n\nSo they could tie the score.',
 'joke_google': [{'type': 'text',
   'text': 'Why are football players excellent at staying cool during a game?\n\nBecause they have so many fans!',
   'extras': {'signature': 'EmAKXgFpFH0TWpojG6Oy2ybIpFxUDX/aocE26h3R1lb6k2Rz5lVEK+tEKt4gA2IlXqic4v9c+61nf0yivwFu+l+pbBEV/w7L8D75UTPjM5zk/CvYsHxCKTyA8JtQSb8BAxI='}}],
 'exp_open': 'The joke works because **“tie”** has two meanings:\n\n- In sports, a **tie** means the game ends with both teams having the same score.\n- A **tie** is also something you can **make with string**.\n\nSo when the joke says, “they brought string so they could tie the score,” it’s a pun: the team is pretending they need string to literally “tie” the score, when really it means they want the game to end in a draw.',
 'exp_google': [{'type': 'text',
   'text': 'This joke relies on a **pun** (a play on words) based on the double meaning of the word

In [ ]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke_open': 'Why did the football team bring string to the game?\n\nSo they could tie the score.', 'joke_google': [{'type': 'text', 'text': 'Why are football players excellent at staying cool during a game?\n\nBecause they have so many fans!', 'extras': {'signature': 'EmAKXgFpFH0TWpojG6Oy2ybIpFxUDX/aocE26h3R1lb6k2Rz5lVEK+tEKt4gA2IlXqic4v9c+61nf0yivwFu+l+pbBEV/w7L8D75UTPjM5zk/CvYsHxCKTyA8JtQSb8BAxI='}}], 'exp_open': 'The joke works because **“tie”** has two meanings:\n\n- In sports, a **tie** means the game ends with both teams having the same score.\n- A **tie** is also something you can **make with string**.\n\nSo when the joke says, “they brought string so they could tie the score,” it’s a pun: the team is pretending they need string to literally “tie” the score, when really it means they want the game to end in a draw.', 'exp_google': [{'type': 'text', 'text': 'This joke relies on a **pun** (a play on words) based on the double meaning of 

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke_open': 'Why did the football team bring string to the game?\n\nSo they could tie the score.', 'joke_google': [{'type': 'text', 'text': 'Why are football players excellent at staying cool during a game?\n\nBecause they have so many fans!', 'extras': {'signature': 'EmAKXgFpFH0TWpojG6Oy2ybIpFxUDX/aocE26h3R1lb6k2Rz5lVEK+tEKt4gA2IlXqic4v9c+61nf0yivwFu+l+pbBEV/w7L8D75UTPjM5zk/CvYsHxCKTyA8JtQSb8BAxI='}}], 'exp_open': 'The joke works because **“tie”** has two meanings:\n\n- In sports, a **tie** means the game ends with both teams having the same score.\n- A **tie** is also something you can **make with string**.\n\nSo when the joke says, “they brought string so they could tie the score,” it’s a pun: the team is pretending they need string to literally “tie” the score, when really it means they want the game to end in a draw.', 'exp_google': [{'type': 'text', 'text': 'This joke relies on a **pun** (a play on words) based on the double meaning of

In [ ]:


config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)



/home/j/miniconda3/envs/langgraph-test/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/j/miniconda3/envs/langgraph-test/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'topic': 'Cricket',
 'joke_open': 'Why did the cricket batter bring a ladder to the match?\n\nBecause he heard the pitch was going to be over his head!',
 'joke_google': [{'type': 'text',
   'text': 'Why do cricket players love to drink tea?\n\nBecause every time they go out to the crease, they get a **break**!',
   'extras': {'signature': 'EmAKXgFpFH0TiBus/P/dXIOxmnX8mMZkXAQeY19mIAQMqwONzN+QpUEpx8RL1StW2K5bB7NNguETXZvW2aQxlZHRUH5RSvisjHqLr3DY9IUhYLbeynHA4FEt4t7J6wpW6zs='}}],
 'exp_open': 'Here’s the joke explained:\n\n- In cricket, the **pitch** is the strip of ground where the bowler delivers the ball and the batter stands.\n- The phrase **“over his head”** usually means something is too difficult to understand.\n- But it can also literally mean something is physically above someone’s head.\n\nSo the joke plays on that double meaning:\n- The batter brought a **ladder** because he thought the pitch would be **too high above him**.\n- That’s silly, because a cricket pitch is flat and 

In [ ]:
workflow.get_state(config2)


StateSnapshot(values={'topic': 'Cricket', 'joke_open': 'Why did the cricket batter bring a ladder to the match?\n\nBecause he heard the pitch was going to be over his head!', 'joke_google': [{'type': 'text', 'text': 'Why do cricket players love to drink tea?\n\nBecause every time they go out to the crease, they get a **break**!', 'extras': {'signature': 'EmAKXgFpFH0TiBus/P/dXIOxmnX8mMZkXAQeY19mIAQMqwONzN+QpUEpx8RL1StW2K5bB7NNguETXZvW2aQxlZHRUH5RSvisjHqLr3DY9IUhYLbeynHA4FEt4t7J6wpW6zs='}}], 'exp_open': 'Here’s the joke explained:\n\n- In cricket, the **pitch** is the strip of ground where the bowler delivers the ball and the batter stands.\n- The phrase **“over his head”** usually means something is too difficult to understand.\n- But it can also literally mean something is physically above someone’s head.\n\nSo the joke plays on that double meaning:\n- The batter brought a **ladder** because he thought the pitch would be **too high above him**.\n- That’s silly, because a cricket pitch 

In [ ]:
workflow.get_state({"configurable": {"thread_id": "2", 'checkpoint_id': '1f1d9b27-6211-6299-8001-8d7253eac817'}})

StateSnapshot(values={'topic': 'Cricket', 'joke_open': 'Why did the cricket batter bring a ladder to the match?\n\nBecause he heard the pitch was going to be over his head!', 'joke_google': [{'type': 'text', 'text': 'Why do cricket players love to drink tea?\n\nBecause every time they go out to the crease, they get a **break**!', 'extras': {'signature': 'EmAKXgFpFH0TiBus/P/dXIOxmnX8mMZkXAQeY19mIAQMqwONzN+QpUEpx8RL1StW2K5bB7NNguETXZvW2aQxlZHRUH5RSvisjHqLr3DY9IUhYLbeynHA4FEt4t7J6wpW6zs='}}]}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1d9b27-6211-6299-8001-8d7253eac817'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-11-06T07:19:53.448087+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1d9b27-4be4-6a9e-8000-1d748aa8b60a'}}, tasks=(PregelTask(id='79fbd922-486c-42a3-4acd-43663df0ca12', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), 

In [ ]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke_open': 'Why did the cricket batter bring a ladder to the match?\n\nBecause he heard the pitch was going to be over his head!', 'joke_google': [{'type': 'text', 'text': 'Why do cricket players love to drink tea?\n\nBecause every time they go out to the crease, they get a **break**!', 'extras': {'signature': 'EmAKXgFpFH0TiBus/P/dXIOxmnX8mMZkXAQeY19mIAQMqwONzN+QpUEpx8RL1StW2K5bB7NNguETXZvW2aQxlZHRUH5RSvisjHqLr3DY9IUhYLbeynHA4FEt4t7J6wpW6zs='}}], 'exp_open': 'Here’s the joke explained:\n\n- In cricket, the **pitch** is the strip of ground where the bowler delivers the ball and the batter stands.\n- The phrase **“over his head”** usually means something is too difficult to understand.\n- But it can also literally mean something is physically above someone’s head.\n\nSo the joke plays on that double meaning:\n- The batter brought a **ladder** because he thought the pitch would be **too high above him**.\n- That’s silly, because a cricket pitch

In [ ]:
workflow.update_state({"configurable": {"thread_id": "2", 'checkpoint_id': '1f1d9b27-6211-6299-8001-8d7253eac817', "checkpoint_ns": ""}}, {'topic':'Tennis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d9b35-93c1-6328-8002-bc1a42c364ec'}}

In [ ]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tennis', 'joke_open': 'Why did the cricket batter bring a ladder to the match?\n\nBecause he heard the pitch was going to be over his head!', 'joke_google': [{'type': 'text', 'text': 'Why do cricket players love to drink tea?\n\nBecause every time they go out to the crease, they get a **break**!', 'extras': {'signature': 'EmAKXgFpFH0TiBus/P/dXIOxmnX8mMZkXAQeY19mIAQMqwONzN+QpUEpx8RL1StW2K5bB7NNguETXZvW2aQxlZHRUH5RSvisjHqLr3DY9IUhYLbeynHA4FEt4t7J6wpW6zs='}}]}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1d9b35-93c1-6328-8002-bc1a42c364ec'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-11-06T07:26:14.467856+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1d9b27-6211-6299-8001-8d7253eac817'}}, tasks=(PregelTask(id='6ee5dd04-89b9-4f4e-9993-da4fba390237', name='generate_explanation', path=('__pregel_pull', 'g

In [ ]:
# Fault TOlerance
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [ ]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [ ]:
#define steps
def step_1(state: CrashState) ->CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

In [ ]:
def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)
    return {"step2":"done"}


In [ ]:
def step_3(state: CrashState) -> CrashState:
    print("Step 3 executed")
    return {"done": True}

In [ ]:
#3 Build the graph

builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")


▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
Step 3 executed


In [ ]:


# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)




🔁 Re-running the graph to demonstrate fault tolerance...

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1da077-8d22-6b38-8003-a8b9c5370638'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-11-06T17:28:25.186170+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1da077-8d20-67f1-8002-47d7ad917efd'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1da077-8d20-67f1-8002-47d7ad917efd'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-06T17:28:25.185259+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1da076-6f04-6a84-8001-db73bb121b6c'}}, tasks=(PregelTask(id='0a2a4267-1bae-3de7-7280-e241b440e13